# Gluten/Velox — 01: Operator Fallback Analysis

## What you will learn
Gluten offloads Spark SQL operators to the native **Velox** engine for faster execution.
However, not every operator is supported — unsupported ones **fall back** to the standard
JVM Spark path transparently, without crashing your job.

Understanding which operators fall back (and why) is essential for:
- Diagnosing unexpected performance results
- Writing queries that maximise Velox offloading
- Knowing the current limits of Gluten 1.6.0 on Spark 4.0

## Architecture recap
```
Spark Logical Plan
      │
      ▼
Gluten Validation Layer  ──(unsupported)──► JVM Spark Executor
      │ (supported)
      ▼
Substrait Plan → Velox Native Engine
```
Every physical operator goes through Gluten's **validator**.
If validation passes → Velox. If not → JVM fallback.


In [ ]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName(os.path.basename(__file__) if '__file__' in dir() else 'notebook')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Gluten: {GLUTEN_ENABLED}")

## Step 1 — Enable Gluten Fallback Logging

By default Gluten suppresses fallback warnings.
We temporarily re-enable them so we can observe which operators fall back and why.

`spark.gluten.sql.columnar.fallbackReporter.enabled` — logs every fallback with reason.


In [ ]:
# Re-enable fallback reporter for this notebook so we can learn from it
spark.conf.set("spark.gluten.sql.columnar.fallbackReporter.enabled", "true")

# Also set log level for the reporter to WARN so it appears in output
log4j = spark.sparkContext._jvm.org.apache.log4j
reporter = "org.apache.gluten.extension.columnar.transition.GlutenFallbackReporter"
log4j.Logger.getLogger(reporter).setLevel(log4j.Level.WARN)
print("Fallback reporter enabled — you will now see WARN messages for each fallback.")

## Step 2 — Create a Sample Dataset

We use a small in-memory dataset so results are instant.
The focus here is on the **query plan**, not performance.


In [ ]:
from pyspark.sql.types import *
import datetime

schema = StructType([
    StructField("id",         LongType()),
    StructField("name",       StringType()),
    StructField("dept",       StringType()),
    StructField("salary",     DoubleType()),
    StructField("hire_date",  DateType()),
    StructField("score",      DoubleType()),
])

data = [
    (1, "Alice",   "Engineering", 95000.0, datetime.date(2019, 3, 1),  8.7),
    (2, "Bob",     "Marketing",   72000.0, datetime.date(2020, 7, 15), 7.2),
    (3, "Carol",   "Engineering", 105000.0,datetime.date(2018, 1, 10), 9.1),
    (4, "Dave",    "HR",          65000.0, datetime.date(2021, 11, 3), 6.8),
    (5, "Eve",     "Engineering", 112000.0,datetime.date(2017, 6, 22), 9.5),
    (6, "Frank",   "Marketing",   78000.0, datetime.date(2022, 2, 8),  7.9),
    (7, "Grace",   "HR",          68000.0, datetime.date(2020, 9, 14), 7.0),
    (8, "Henry",   "Engineering", 98000.0, datetime.date(2019, 5, 30), 8.2),
]

df = spark.createDataFrame(data, schema)
df.createOrReplaceTempView("employees")
df.show()
print(f"Schema: {df.schema.simpleString()}")

## Step 3 — Inspect the Physical Plan: Simple Aggregation

`explain(mode="formatted")` shows the **physical plan** with Gluten annotations.

Look for:
- `GlutenColumnarToRow` — data coming back from Velox to JVM
- `VeloxColumnarToRow` — same, Velox-specific wrapper
- `*(1) HashAggregate` prefixed with `GlutenPlan` — offloaded to Velox
- Plain `HashAggregate` without Gluten prefix — running on JVM

> **Tip:** operators inside `WholeStageCodegen` are JVM; operators tagged
> with Gluten/Velox names are native.


In [ ]:
# Simple aggregation — highly likely to be offloaded to Velox
agg_df = df.groupBy("dept").agg(
    F.count("*").alias("headcount"),
    F.avg("salary").alias("avg_salary"),
    F.max("score").alias("top_score")
)

print("=" * 70)
print("PHYSICAL PLAN — Simple Aggregation")
print("=" * 70)
agg_df.explain(mode="formatted")

## Step 4 — Window Functions (Known Fallback in Gluten 1.6.0)

Window functions that require a **shuffle exchange** currently fall back to JVM
in Gluten 1.6.0 due to an incompatibility in `ColumnarBatchSerializerInstanceImpl`.

Observe the plan — you will see `Exchange` nodes that are NOT offloaded.

> **Why does this matter?**  
> If your pipeline is heavy on window functions, Gluten may not give you
> the expected speedup. This is tracked upstream and expected to be fixed
> in Gluten 1.7+.


In [ ]:
from pyspark.sql.window import Window

# Window with PARTITION BY + ORDER BY — triggers shuffle Exchange
w = Window.partitionBy("dept").orderBy(F.desc("salary"))

window_df = df.withColumn("rank_in_dept",   F.rank().over(w)) \
              .withColumn("salary_pct",      F.percent_rank().over(w)) \
              .withColumn("dept_avg_salary", F.avg("salary").over(Window.partitionBy("dept")))

print("=" * 70)
print("PHYSICAL PLAN — Window Function (observe Exchange nodes)")
print("=" * 70)
window_df.explain(mode="formatted")

# This will trigger the fallback warning in the log
# We wrap in try/except since large shuffles can fail in Gluten 1.6.0
try:
    window_df.show(5)
    print("\nWindow query completed.")
except Exception as e:
    print(f"\nFallback triggered (expected in Gluten 1.6.0): {type(e).__name__}")
    print("Use vanilla Spark for heavy window workloads until Gluten 1.7+")

## Step 5 — Scan which nodes are Gluten vs JVM

We can programmatically inspect the executed plan and classify each node.
This is useful for automated regression testing of offload rates.


In [ ]:
def analyze_plan(df, query_name):
    """
    Walk the physical plan and classify nodes as:
      - VELOX   : processed by Gluten/Velox native engine
      - JVM     : processed by standard Spark JVM execution
      - EXCHANGE: shuffle/broadcast node (potential fallback point)
    """
    plan_str = df._jdf.queryExecution().executedPlan().toString()
    lines = plan_str.split("\n")

    velox_nodes  = [l.strip() for l in lines if "Gluten" in l or "Velox" in l or "Columnar" in l]
    jvm_nodes    = [l.strip() for l in lines if "WholeStageCodegen" in l or "HashAggregate" in l]
    exchange_nodes = [l.strip() for l in lines if "Exchange" in l]

    print(f"\n{'='*60}")
    print(f"Query: {query_name}")
    print(f"{'='*60}")
    print(f"  Velox/Gluten nodes : {len(velox_nodes)}")
    print(f"  JVM nodes          : {len(jvm_nodes)}")
    print(f"  Exchange nodes     : {len(exchange_nodes)}")

    if velox_nodes:
        print("\n  -- Velox nodes --")
        for n in velox_nodes[:5]:
            print(f"    {n[:80]}")
    if exchange_nodes:
        print("\n  -- Exchange nodes (watch for fallback) --")
        for n in exchange_nodes[:3]:
            print(f"    {n[:80]}")
    return len(velox_nodes), len(jvm_nodes)

v1, j1 = analyze_plan(agg_df,    "Simple Aggregation")
v2, j2 = analyze_plan(df.filter(F.col("salary") > 90000), "Filter + Scan")
v3, j3 = analyze_plan(df.join(df.select("dept","salary").groupBy("dept").agg(F.avg("salary").alias("dept_avg")), "dept"), "Join + Agg")

print(f"\nSummary:")
print(f"  Aggregation  — Velox: {v1}, JVM: {j1}")
print(f"  Filter       — Velox: {v2}, JVM: {j2}")
print(f"  Join + Agg   — Velox: {v3}, JVM: {j3}")

## Step 6 — Operators That Velox Handles Well

These are the **sweet spot** operations for Gluten/Velox — expect 2–5x speedup:

| Operation | Velox Support | Notes |
|---|---|---|
| `filter` / `WHERE` | ✅ Full | Vectorised predicate evaluation |
| `groupBy` + `agg` | ✅ Full | HashAgg fully native |
| `sort` | ✅ Full | Native sort |
| `join` (hash/sort-merge) | ✅ Full | Columnar hash join |
| Parquet scan | ✅ Full | Native Parquet reader |
| `project` (arithmetic, string) | ✅ Most | Some complex string ops fall back |
| Window (no shuffle) | ⚠️ Partial | Simple frames ok, shuffle exchange falls back |
| UDFs (Python) | ❌ Always fallback | Python UDFs are JVM-only |
| `ANSI mode` operations | ❌ Fallback | Disable with `spark.sql.ansi.enabled=false` |

Let's verify filter + project offloading:


In [ ]:
# Operations Velox handles very well
well_supported = (
    df.filter((F.col("salary") > 80000) & (F.col("score") >= 8.0))
      .select(
          "name", "dept",
          (F.col("salary") * 1.10).alias("salary_with_raise"),
          F.round(F.col("score"), 1).alias("score_rounded"),
          F.upper(F.col("dept")).alias("dept_upper")
      )
      .orderBy(F.desc("salary_with_raise"))
)

print("PHYSICAL PLAN — Filter + Project + Sort (Velox sweet spot)")
well_supported.explain(mode="formatted")
well_supported.show()

## Step 7 — Python UDFs Always Fall Back

Python UDFs execute in a separate Python process — Velox cannot run Python code.
Any DataFrame that uses a Python UDF will fall back to JVM for that operation.

**Mitigation:** Use built-in Spark functions (`F.`) instead of Python UDFs wherever possible.
If you must use Python logic, use **Pandas UDFs** (vectorized) — they still fall back
but are much faster than row-at-a-time Python UDFs.


In [ ]:
# Python UDF — always JVM, never Velox
@F.udf(returnType=DoubleType())
def bonus_udf(salary, score):
    """Row-at-a-time Python UDF — always falls back to JVM."""
    return salary * (score / 10.0) * 0.15

# Equivalent using native Spark functions — stays in Velox
bonus_native = df.withColumn("bonus_native",
    F.col("salary") * (F.col("score") / 10.0) * 0.15)

bonus_udf_df = df.withColumn("bonus_udf", bonus_udf(F.col("salary"), F.col("score")))

print("PLAN — Native functions (stays in Velox):")
bonus_native.explain(mode="formatted")

print("\nPLAN — Python UDF (falls back to JVM for UDF evaluation):")
bonus_udf_df.explain(mode="formatted")

# Time comparison
import time

t0 = time.time()
bonus_native.collect()
t_native = time.time() - t0

t0 = time.time()
bonus_udf_df.collect()
t_udf = time.time() - t0

print(f"\nPerformance comparison (small dataset — illustrative):")
print(f"  Native functions : {t_native:.3f}s")
print(f"  Python UDF       : {t_udf:.3f}s")
print(f"  Overhead factor  : {t_udf/t_native:.1f}x  (more visible on large datasets)")

## Step 8 — Summary: Fallback Decision Tree

```
Is the operator a Python UDF?
  YES → Always JVM fallback
  NO  ↓
Is ANSI mode enabled?
  YES → Many operators fall back (set spark.sql.ansi.enabled=false)
  NO  ↓
Is it a Window function with shuffle Exchange?
  YES → Falls back in Gluten 1.6.0 (tracked bug)
  NO  ↓
Is it a standard aggregation / filter / join / sort / scan?
  YES → Offloaded to Velox ✅
```

### Key takeaways
1. **Use built-in functions** (`F.col`, `F.sum`, `F.when`, etc.) — they offload to Velox
2. **Avoid Python UDFs** on hot paths — use Pandas UDFs if you need Python logic
3. **Disable ANSI mode** (`spark.sql.ansi.enabled=false`) — already in spark-defaults.conf
4. **Window functions** with large partitions currently fall back — use them selectively
5. **Check `explain()`** regularly — it tells you exactly what Velox is running

### How to measure your offload rate
Run `analyze_plan()` (defined in Step 5) on your production queries.
A healthy Gluten workload should have **> 80% Velox nodes** in the plan.


In [ ]:
# Final: disable fallback reporter again to keep logs clean in other notebooks
log4j.Logger.getLogger(reporter).setLevel(log4j.Level.ERROR)
print("Fallback reporter set back to ERROR level.")
print("\nNotebook complete. Key files:")
print("  - Gluten docs: https://gluten.apache.org/docs/v1.6.0/get-started/Velox.html")
print("  - Configuration: conf/spark-defaults.conf")
print("  - Next notebook: 02_velox_performance_deep_dive.ipynb")